In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

my_ID = 2290
np.random.seed(my_ID)
np.set_printoptions(precision=5,suppress=True)# reduced display precision on numpy arrays

In [2]:
data_train = pd.read_csv('fashion-mnist_train.csv')
data_test = pd.read_csv('fashion-mnist_test.csv')

In [3]:
# create training and validation set
X_train = data_train.iloc[:, 1:].to_numpy()
t_train = data_train.iloc[:,0].to_numpy()
X_train, X_valid, t_train, t_valid = train_test_split(X_train, t_train, test_size=0.2, random_state=my_ID)
X_train = X_train / 255
X_valid = X_valid / 255
print(f"x_train Shape: {X_train.shape}, x_train Type:{type(X_train)})")
print(f"x_valid Shape: {t_valid.shape}, x_valid Type:{type(t_valid)})")

# create test set
X_test = data_test.iloc[:, 1:].to_numpy()
t_test = data_test.iloc[:,0].to_numpy()
X_test = X_test / 255
print(f"x_test Shape: {X_test.shape}, x_test Type:{type(X_test)})")

x_train Shape: (48000, 784), x_train Type:<class 'numpy.ndarray'>)
x_valid Shape: (12000,), x_valid Type:<class 'numpy.ndarray'>)
x_test Shape: (10000, 784), x_test Type:<class 'numpy.ndarray'>)


Small sample data from data_banknote_authentication.txt


In [ ]:
sc = StandardScaler()

dataset = pd.read_csv(
    "data_banknote_authentication.txt",
    header=None)
X_data = dataset.iloc[:, :-1].values
train = dataset.iloc[:, -1].values

rand_seed = 20

# split into train, test and validation set and standardize
X_train, X_test, t_train, t_test = train_test_split(X_data, train, test_size=0.2, random_state=rand_seed)
X_train, X_valid, t_train, t_valid = train_test_split(X_train, t_train, test_size=0.25, random_state=rand_seed)
X_train = sc.fit_transform(X_train)
X_valid = sc.transform(X_valid)
X_test = sc.transform(X_test)

In [ ]:
# some convenient code
def l_ce(t, y_pred):
    y_pred[y_pred == 1] = 0.99999999999999999
    y_pred[y_pred == 0] = 0.00000000000000001
    return np.dot(-1 * t, np.log(y_pred)) - np.dot((1 - t), np.log(1 - y_pred))

def l_ce_softmax(t, y_pred):
    t = np.array(t, dtype=int)   # make sure it's integer
    return -np.mean(np.log(y_pred[np.arange(len(t)), t] + 1e-12))

def l_ce_softmax_batch(t, y_pred):
    # y_batch: shape (num_classes, batch_size)
    # t_batch: shape (batch_size,) integers
    return -np.mean([np.log(y_pred[t_i, i] + 1e-12) for i, t_i in enumerate(t)])

def sigmoid(z):
    return (1/ (1 + np.exp(-z)))

def relu(h):
    h[h < 0] = 0
    return h


def d_relu(g):
    # relu derivative
    g[g >= 0] = 1
    g[g < 0] = 0
    return g

# def d_relu(g):
#     return (g > 0).astype(float)

def new_w(w, w_J):
    alpha = 0.01
    w_new = w - np.dot(alpha, w_J)
    return w_new


def neural_net_SGD(x, t, Ws):
    L = len(Ws)

    zs = []
    hs = []

    # fwd
    z1 = np.dot(Ws[0], np.r_[1, x.T][:, None])
    # print("z1: {0}\n".format(z1))
    h1 = relu(z1.copy())
    # print("h1: {0}\n".format(h1))
    zs.append(z1)
    hs.append(h1)
    # z2 = np.dot(W2, np.c_[1, h1.T].T)
    # # print("z2: {0}\n".format(z2))
    # h2 = relu(z2.copy())
    # # print("h2: {0}\n".format(h2))

    # middle layers
    h_prev = h1
    for l in range(1,L-1):
        z = np.dot(Ws[l], np.c_[1, h_prev.T].T)
        h = relu(z.copy())
        zs.append(z)
        hs.append(h)
        h_prev = h

    zd = np.dot(Ws[-1], np.c_[1, h_prev.T].T)
    zs.append(zd)
    
    # print('zs = ',zs)
    # print("z3: {0}\n".format(z3))
    # y = 1 / (1 + np.exp(-zd)) # for binary
    exp_z = np.exp(zd - np.max(zd))   # numerical stability
    y = exp_z / np.sum(exp_z)
    hs.append(y) # y is h_d
    #print('hs = ',hs)
    # print("y: {0}\n".format(y))

    # bwd
    # z is 1 shorter than w cus (d-1) d = 3
    w_Js = []
    z_Js = []
    t_one_hot = np.zeros_like(y)
    t_one_hot[int(t)] = 1
    dj_dz_d = -t_one_hot + y  # gradient of cross-entropy loss dJ/dz
    #z_Js.append(dj_dz_d)
    wd_J = dj_dz_d @ np.c_[1, h_prev.T]
    w_Js.append(wd_J)
    zdm1_J = np.multiply(d_relu(zs[-2]), Ws[-1][:, 1:].T @ dj_dz_d)  # element wise multiplication
    z_Js.insert(0,zdm1_J)

    # middle layers (if more than one)
    for l in reversed(range(1, L-1)):
        # dz of current layer
        dz_current = np.multiply(d_relu(zs[l-1]), np.dot(Ws[l][:, 1:].T, z_Js[0]))
        #print("dz_current = ", dz_current)
        z_Js.insert(0, dz_current)  # prepend so z_Js order matches layers
        
        # gradient for weights at this layer
        h_input = hs[l-1] if l > 1 else h1
        #print("h_input = ",h_input)
        w_grad = np.dot(z_Js[1], np.c_[1, h_input.T])
        #print("w_grad = ", w_grad)
        w_Js.insert(0, w_grad)  # prepend so w_Js order matches Ws

    
    w1_J = np.dot(z_Js[0], np.r_[1, x.T][None, :])
    w_Js.insert(0, w1_J)
    
    # print('w_Js = ', w_Js)
    # print('z_Js = ',z_Js)
    # new weights
    # W1_new = new_w(Ws[0], w_Js[0])
    # W2_new = new_w(Ws[1], w_Js[1])
    # W3_new = new_w(Ws[2], w_Js[2])

    lam = 0.001232  # weight decay strength
    #print("z_Js = ", z_Js)
    #print("w_Js = ", w_Js)
    for i in range(len(Ws)):
        w_Js[i][:,1:] += lam * Ws[i][:,1:]

    w_new = []
    for i in range(len(Ws)):
        w_new.append(new_w(Ws[i], w_Js[i]))

    return y, w_new

def neural_net_batch(x, t, Ws, sample_per_batch):
    L = len(Ws)

    zs = []
    hs = []

    # fwd
    Xb = np.c_[np.ones((sample_per_batch, 1)), x]  # shape: (batch_size, n_features+1)
    z1 = Xb @ Ws[0].T  # Ws[0]: (n_hidden, n_features+1), result: (batch_size, n_hidden)
    # print("z1: {0}\n".format(z1))
    h1 = relu(z1.copy())
    # print("h1: {0}\n".format(h1))
    zs.append(z1)
    hs.append(h1)
    # z2 = np.dot(W2, np.c_[1, h1.T].T)
    # # print("z2: {0}\n".format(z2))
    # h2 = relu(z2.copy())
    # # print("h2: {0}\n".format(h2))

    # middle layers
    h_prev = h1
    for l in range(1,L-1):
        H_prev_b = np.c_[np.ones((sample_per_batch, 1)), h_prev]  # add bias
        z = H_prev_b @ Ws[l].T  # shape: (batch_size, n_hidden_next)
        h = relu(z.copy())
        zs.append(z)
        hs.append(h)
        h_prev = h

    zd = np.dot(Ws[-1], np.c_[np.ones(sample_per_batch), h_prev.T].T)
    zs.append(zd)
    
    # print('zs = ',zs)
    # print("z3: {0}\n".format(z3))
    # y = 1 / (1 + np.exp(-zd)) # for binary
    exp_z = np.exp(zd - np.max(zd, axis=1, keepdims=True))  # subtract max per sample
    y = exp_z / np.sum(exp_z, axis=1, keepdims=True)         # softmax per sample
    hs.append(y) # y is h_d
    #print('hs = ',hs)
    # print("y: {0}\n".format(y))

    # bwd
    # z is 1 shorter than w cus (d-1) d = 3
    w_Js = []
    z_Js = []
    t_one_hot = np.zeros_like(y)
    t_one_hot[np.arange(sample_per_batch), t] = 1
    dj_dz_d = (y - t_one_hot) / sample_per_batch
    #z_Js.append(dj_dz_d)
    H_prev_b = np.c_[np.ones((sample_per_batch, 1)), h_prev]  # batch_size x n_prev+1
    wd_J = dj_dz_d.T @ H_prev_b  # shape: n_classes x (n_prev + 1)
    w_Js.append(wd_J)
    zdm1_J = np.multiply(d_relu(zs[-2]), Ws[-1][:, 1:].T @ dj_dz_d)  # element wise multiplication
    z_Js.insert(0,zdm1_J)

    # middle layers (if more than one)
    for l in reversed(range(1, L-1)):
        # dz of current layer
        dz_current = np.multiply(d_relu(zs[l-1]), np.dot(Ws[l][:, 1:].T, z_Js[0]))
        #print("dz_current = ", dz_current)
        z_Js.insert(0, dz_current)  # prepend so z_Js order matches layers
        
        # gradient for weights at this layer
        h_input = hs[l-1] if l > 1 else h1
        #print("h_input = ",h_input)
        H_input_b = np.c_[np.ones((sample_per_batch, 1)), h_input]
        w_grad = dz_current.T @ H_input_b
        #print("w_grad = ", w_grad)
        w_Js.insert(0, w_grad)  # prepend so w_Js order matches Ws

    
    w1_J = np.dot(z_Js[0], np.r_[np.ones(sample_per_batch), x.T][None, :])
    w_Js.insert(0, w1_J)
    
    # print('w_Js = ', w_Js)
    # print('z_Js = ',z_Js)
    # new weights
    # W1_new = new_w(Ws[0], w_Js[0])
    # W2_new = new_w(Ws[1], w_Js[1])
    # W3_new = new_w(Ws[2], w_Js[2])

    lam = 0.001232  # weight decay strength
    #print("z_Js = ", z_Js)
    #print("w_Js = ", w_Js)
    for i in range(len(Ws)):
        w_Js[i][:,1:] += lam * Ws[i][:,1:]

    w_new = []
    for i in range(len(Ws)):
        w_new.append(new_w(Ws[i], w_Js[i]))

    return y, w_new

In [ ]:
def initialize_weights_random(num_hidden_layers, input_size=784, hidden_size_H=80, output_size=10):
    Ws = []

    # Input → first hidden
    W_prev = np.random.randn(hidden_size_H, input_size + 1) * 0.01
    Ws.append(W_prev)

    # Hidden → hidden
    for _ in range(1, num_hidden_layers):
        W_hidden = np.random.randn(hidden_size_H, hidden_size_H + 1) * 0.01
        Ws.append(W_hidden)

    # Hidden → output
    W_out = np.random.randn(output_size, hidden_size_H + 1) * 0.01
    Ws.append(W_out)

    return Ws

def Gradient_desent_SGD(Ws):
    # Setup list to help graph later
    epoch = []
    err_valid = []
    err_train = []

    # setup for early stop
    best_weights = None
    best_ve = float('inf')
    no_improve_count = 0
    n = 5
    m = 5

    for i in range(100): # epoch
        epoch.append(i)
        print(f'Epoch {i}')

        # SGD shaffle rows each time
        temp_arr = np.c_[X_train, t_train] # This concatenate array along the column axis
        np.random.shuffle(temp_arr)
        X = temp_arr[:, :-1]
        t = temp_arr[:, -1]

        y_train = [] # output
        for j in range(len(X_train)): # iterate through every sample
            # update the weights each iteration
            
            y, Ws = neural_net_SGD(X[j, :], t[j], Ws)
            #print('y=' ,y)
            y_train.append(y.flatten())
            #print ('y_train =', y_train)
        #print(Ws)
        y_train = np.array(y_train) # py list to np array
        # print('shape of y_train np = ', y_train.shape)
        # print("First 5 samples of y_train:\n", y_train[:10])
        # training cross entropy
        ce_loss_train = l_ce_softmax(t, y_train)
        #print('ce_loss=',ce_loss_train)
        err_train.append(ce_loss_train)
        
        # validation cross entropy
        y_valid = []
        for j in range(len(X_valid)):  # iterate through every sample
            # use trained weights each iteration
            
            y, _WS = neural_net_SGD(X_valid[j, :], t_valid[j], Ws)
            y_valid.append(y.flatten())

        y_valid = np.array(y_valid)
        # print("First 5 samples of y_valid:\n", y_train[:10])
        ce_loss_valid = l_ce_softmax(t_valid, y_valid)
        err_valid.append(ce_loss_valid)

        # for signoid obly
        # # if output prob distribution > 0.5, classify it as class 1 else 0
        # y_class = np.where(y_valid > 0.5, 1, 0)
        # misclass_rate = 1 - accuracy_score(t_valid, y_class)

        # softmax
        pred_classes = np.argmax(y_valid, axis=1)
        accuracy = np.mean(pred_classes == t_valid)

        print(f"Validation accuracy: {accuracy * 100:.2f}%")
        # for i in range(len(t_valid)):
        #     print(f"Sample {i}: Predicted = {pred_classes[i]}, Actual = {int(t_valid[i])}")

        #early stop
        # m = 10
        # if len(err_valid) >= m:
        #     if ce_loss_valid > min(err_valid[-m:]):  # compare to last m epochs
        #         print(f"Early stopping at epoch {i}")
                # break
        # if i % n == 0:

        #     if ce_loss_valid < best_ve:
        #         best_ve = ce_loss_valid
        #         best_weights = [W.copy() for W in Ws]  # save the best weights
        #         no_improve_count = 0
        #     else:
        #         no_improve_count += n

        #     if no_improve_count >= m * n:
        #         print(f"Early stopping at iteration {i}")
        #         break

    pred_classes = np.argmax(y_valid, axis=1)
    accuracy = np.mean(pred_classes == t_valid)

    print(f"Validation accuracy: {accuracy * 100:.2f}%")
        
    # plot learning curve
    plt.plot(epoch, err_train, color='red', label='training')
    plt.plot(epoch, err_valid, color='green', label='validation')
    plt.title('learning curve')
    plt.legend(loc='upper right')
    plt.xlabel('epoch')
    plt.ylabel('cross entropy loss')
    plt.show()
    print("\n")

    # calc validation, training error
    print("err_train: {0}".format(err_train[-1]))
    print("err_valid: {0}".format(err_valid[-1]))

def Gradient_desent_Batch(Ws):
    # Setup list to help graph later
    epoch = []
    err_valid = []
    err_train = []

    # setup for early stop
    best_weights = None
    best_ve = float('inf')
    no_improve_count = 0
    n = 5
    m = 5

    for i in range(100): # epoch
        epoch.append(i)
        print(f'Epoch {i}')

        # SGD shaffle rows each time
        temp_arr = np.c_[X_train, t_train] # This concatenate array along the column axis
        np.random.shuffle(temp_arr)
        X = temp_arr[:, :-1]
        t = temp_arr[:, -1]

        y_train = [] # output
        for j in range(len(X_train)): # iterate through every sample
            # update the weights each iteration
            
            y, Ws = neural_net_SGD(X[j, :], t[j], Ws)
            #print('y=' ,y)
            y_train.append(y.flatten())
            #print ('y_train =', y_train)
        #print(Ws)
        y_train = np.array(y_train) # py list to np array
        # print('shape of y_train np = ', y_train.shape)
        # print("First 5 samples of y_train:\n", y_train[:10])
        # training cross entropy
        ce_loss_train = l_ce_softmax(t, y_train)
        #print('ce_loss=',ce_loss_train)
        err_train.append(ce_loss_train)
        
        # validation cross entropy
        y_valid = []
        for j in range(len(X_valid)):  # iterate through every sample
            # use trained weights each iteration
            
            y, _WS = neural_net_SGD(X_valid[j, :], t_valid[j], Ws)
            y_valid.append(y.flatten())

        y_valid = np.array(y_valid)
        # print("First 5 samples of y_valid:\n", y_train[:10])
        ce_loss_valid = l_ce_softmax(t_valid, y_valid)
        err_valid.append(ce_loss_valid)

        # for signoid obly
        # # if output prob distribution > 0.5, classify it as class 1 else 0
        # y_class = np.where(y_valid > 0.5, 1, 0)
        # misclass_rate = 1 - accuracy_score(t_valid, y_class)

        # softmax
        pred_classes = np.argmax(y_valid, axis=1)
        accuracy = np.mean(pred_classes == t_valid)

        print(f"Validation accuracy: {accuracy * 100:.2f}%")


def main():

    number_of_hidden_layer = 3
    #Ws = initialize_weights(number_of_hidden_layer)
    Ws = initialize_weights_random(number_of_hidden_layer)
    # Gradient_desent_SGD(Ws)


    
        

if __name__ == '__main__':
    main()

Epoch 0
Validation accuracy: 75.29%
Epoch 1
Validation accuracy: 83.17%
Epoch 2
Validation accuracy: 81.58%
Epoch 3


KeyboardInterrupt: 